# OE2 — Parte B: Generalización mediante Aprendizaje Basado en Pares

**Tesis:** Colmena Inteligente — Sistema de Monitoreo Multimodal de Colmenas  
**Objetivo Específico 2, Parte B:** Generalización cross-dataset mediante clasificación de similitud (xnorC)  
**Inspiración:** BeeTogether (2024) — extendido con aportes originales  
**Foco de este notebook:** Solo modalidad de audio (xnorC-AUD). La modalidad tabular (xnorC-TAB, xnorC-HYB) queda para una etapa posterior.

---

## Propósito de este notebook

Este notebook es el **plan maestro** de OE2-B. No ejecuta experimentos — eso lo hacen los notebooks OE2B-01 a OE2B-07. Lo que hace este notebook es:

1. Describir todos los datasets disponibles y su estado de uso
2. Explicar el marco teórico (ClassiC vs. PairC vs. xnorC)
3. Detallar todos los experimentos planificados con sus parámetros
4. Documentar las decisiones de diseño y los riesgos
5. Servir como referencia durante las reuniones con el tutor

**Pipeline completo de OE2-B:**
```
OE2B-00  → Este notebook (plan maestro)
OE2B-01  → Auditoría y preprocesamiento de NU-HIVE
OE2B-02  → Auditoría y preprocesamiento de Yang (para Parte B)
OE2B-03  → Construcción del dataset de pares (Yang + NU-HIVE)
OE2B-04  → Extracción de features de audio (Mel, MFCC, FFT)
OE2B-05  → Entrenamiento xnorC (experimentos B1, B2, B3)
OE2B-06  → Evaluación cross-hive (k-hive-fold)
OE2B-07  → Evaluación cross-country (UrBAN)
```

---
# 1. Marco Teórico — ¿Por qué xnorC?

## 1.1. El problema de generalización entre colmenas

Una colmena tiene una **"firma acústica" propia**: la combinación de su ubicación geográfica, raza de abejas, diseño de la caja, tipo de micrófono y estación del año produce un espectro de frecuencias único. Un modelo clásico entrenado en las colmenas A, B y C aprende a asociar "esta frecuencia particular = colmena A en estado QR". Cuando se le presenta una colmena D no vista, el modelo no reconoce ningún patrón familiar y falla.

Este fenómeno fue demostrado empíricamente por **BeeTogether (2024)**:
- Exactitud en validación aleatoria (mismas colmenas en train y test): **~98%**
- Exactitud en validación k-hive-fold (una colmena entera excluida): **~50% (azar)**

La diferencia entre ambas validaciones es la medida del **gap de generalización**.

## 1.2. Los tres clasificadores de BeeTogether

### ClassiC — Clasificador Clásico (línea base)

Recibe el espectrograma de un segmento de audio y predice directamente si la reina está presente (QR) o ausente (QL).

```
Entrada:  [espectrograma_A]  →  CNN  →  QR / QL
```

**Por qué falla:** aprende la firma acústica de cada colmena, no el estado de la reina. Su representación interna codifica *"esto suena a la colmena A en estado QR"* en lugar de *"esto suena a una colmena sin reina en general"*.

### PairC — Clasificador de Pares (intento intermedio)

Recibe dos espectrogramas de la misma colmena y predice la etiqueta de cada uno por separado.

```
Entrada:  [espectrograma_A, espectrograma_B]  →  CNN  →  QR/QL, QR/QL
```

**Por qué falla:** sigue prediciendo la etiqueta absoluta (QR o QL), por lo que continúa aprendiendo características de cada colmena específica.

### xnorC — Clasificador XNOR (la propuesta que funciona)

Cambia radicalmente la pregunta: en lugar de "¿qué estado tiene esta muestra?", pregunta **"¿estas dos muestras tienen el mismo estado o estados distintos?"**

```
Entrada:  [espectrograma_A, espectrograma_B]  →  Red Siamesa  →  IGUAL / DIFERENTE
```

La etiqueta de salida es binaria:
- **IGUAL (1):** ambas muestras QR, o ambas QL
- **DIFERENTE (0):** una QR y la otra QL

**Por qué funciona:** determinar si dos momentos de la *misma* colmena tienen el mismo estado es posible aunque nunca se haya visto esa colmena antes. El modelo aprende a detectar *cambios internos* dentro de una colmena, no características absolutas de ninguna clase.

> 💡 **Analogía:** un médico que nunca vio a un paciente puede comparar dos electrocardiogramas del mismo paciente y decir "estos dos son iguales" o "este es diferente al anterior", incluso sin conocer al paciente de antemano. xnorC hace exactamente eso con el audio de la colmena.

## 1.3. ¿Es esto aprendizaje contrastativo?

No exactamente. El aprendizaje contrastativo clásico (SimCLR, BYOL, MoCo) es **autosupervisado** — no necesita etiquetas. xnorC usa pares **etiquetados** (necesita saber el estado de cada elemento para calcular si el par es IGUAL o DIFERENTE). El término correcto es **aprendizaje supervisado de similitud** (*supervised similarity learning*), implementado con una arquitectura de **red siamesa** (dos ramas que comparten pesos).

## 1.4. ¿Cómo convierte IGUAL/DIFERENTE en QR/QL?

El modelo xnorC necesita una **muestra de referencia** con etiqueta conocida (provista por una inspección del apicultor). La conversión es:

| Etiqueta referencia | Predicción xnorC | Etiqueta asignada |
|---|---|---|
| QR | IGUAL | QR |
| QR | DIFERENTE | QL |
| QL | IGUAL | QL |
| QL | DIFERENTE | QR |

La referencia es el "ancla" que orienta las predicciones de similitud hacia etiquetas absolutas.

---
# 2. Datasets

## 2.1. Resumen general

| Dataset | País | Año | Colmenas | Etiquetado | Audio crudo | Uso en OE2-B |
|---|---|---|---|---|---|---|
| Yang (SBCM) | EE.UU. (California) | 2017 | 4 | Continuo | ✅ | ✅ Entrenamiento |
| NU-HIVE | UK/Europa | 2017–2018 | 2 | Puntual (4 días) | ✅ | ✅ Entrenamiento adicional |
| OSBH | Multi-país | Varios | Muchas | Variable | ✅ | ❌ Excluido (ver 2.4) |
| harshkumar1711 | Desconocido | 2017–2018 | — | — | ✅ | ❌ Excluido (ver 2.5) |
| UrBAN 2021 | Canadá (Montreal) | 2021 | 7 (con sensor) | Puntual (inspecciones) | ✅ | ✅ Prueba cross-country |
| MSPB 2.0 | Argentina | 2022–2023 | Varias | Puntual | ❌ Solo features | 🔜 Fase posterior (xnorC-TAB) |

---

## 2.2. Yang — Smart Bee Colony Monitor (SBCM)

**Fuente:** Kaggle — Anna J. Yang  
**DOI/URL:** https://www.kaggle.com/datasets/annajyang/beehive-sounds  
**Período:** 2017  
**Ubicación:** California, EE.UU.

### Estructura de audio
- **4 colmenas** monitoreadas
- **6 archivos por hora** por colmena (un archivo cada ~10 minutos)
- **Duración por archivo:** 1 minuto exacto
- **Cobertura:** continua a lo largo de semanas
- **Etiquetado:** continuo — cada hora tiene etiqueta QR o QL
- **Clases:** QR (queen right = reina presente) / QL (queenless = reina ausente)

### Estrategia de agregación temporal para xnorC
Los 6 archivos de 1 minuto por hora se promedian a nivel de features para obtener **1 representación por hora**. Esto es consistente con la granularidad de los sensores ambientales y facilita la comparación entre datasets.

### Rol en OE2-B
Dataset principal de entrenamiento. Todas las colmenas de Yang se usan para construir los pares de entrenamiento. Yang es el "dominio fuente" del que el modelo debe generalizar.

---

## 2.3. NU-HIVE — Audio-Based Identification of Beehive States

**Fuente:** Zenodo — Nolasco et al. (2019)  
**DOI:** https://doi.org/10.5281/zenodo.2667806  
**Período:** 2017–2018  
**Ubicación:** Europa (Reino Unido)

### Estructura de audio
- **2 colmenas** (Colmena 1 y Colmena 3)
- **Grabaciones en 4 días específicos** (no continuo)
- **Frecuencia:** 1 archivo cada 10 minutos
- **Duración por archivo:** ~9 minutos 52 segundos (~592 segundos)

### Detalle de días y etiquetas

| Colmena | Fecha | Estado | Notas |
|---|---|---|---|
| Colmena 1 | 31-05-2018 | QL (sin reina) | Día completo de grabación |
| Colmena 1 | 12-06-2018 | QR (reina activa) | Día completo de grabación |
| Colmena 3 | 14-07-2017 | QL (sin reina) | Día completo de grabación |
| Colmena 3 | 28-07-2017 | QR (reina activa) | Día completo de grabación |

### Consideraciones importantes
- El etiquetado es **a nivel de día entero**, no por archivo individual. Todos los archivos de un día tienen la misma etiqueta.
- Con 4 días de grabación, el dataset tiene ~144 archivos de ~10 min cada uno.
- Es un subconjunto del dataset *"To Bee or Not to Bee"*, pero con las etiquetas QR/QL correctas para nuestro objetivo.

### Estrategia de agregación temporal para xnorC
Los archivos de 10 minutos se promedian a nivel de features en ventanas de 1 hora para alinearlos con Yang (6 archivos Yang × 1 min = 6 min de audio por hora; en NU-HIVE 1 archivo × ~10 min ≈ compatible). Se usará la hora del timestamp del archivo para agrupar.

### Rol en OE2-B
Dataset de entrenamiento adicional. Aporta diversidad geográfica al entrenamiento (colmenas europeas vs. Yang californiano). Dos colmenas × 2 estados × 1 día cada uno = 4 días de datos de alta calidad.

---

## 2.4. OSBH — Open Source Beehives Project

**Fuente:** Zenodo  
**URL:** https://zenodo.org/records/321345  
**Período:** Varios años  
**Ubicación:** Multi-país (Norteamérica, Australia, Europa)

### Estructura
- Carpeta **"Active"**: 304 archivos de ~3:31 minutos
- Carpeta **"Missing Queen"**: **1 solo archivo** de 15 segundos

### ❌ Decisión: EXCLUIDO

**Razón:** El desbalance extremo entre clases hace este dataset inutilizable para entrenamiento supervisado. Con un único ejemplo de la clase "Missing Queen" (15 segundos de audio), no es posible:
1. Entrenar ningún modelo que aprenda la clase QL
2. Construir pares balanceados IGUAL/DIFERENTE
3. Validar con k-hive-fold (no hay suficientes colmenas en la clase negativa)

**Documentación:** Esta exclusión debe mencionarse en la tesis como parte de la selección de datasets.

---

## 2.5. harshkumar1711 — BeeHive Audio Dataset with/without Queen

**Fuente:** Kaggle — harshkumar1711  
**URL:** https://www.kaggle.com/datasets/harshkumar1711/beehive-audio-dataset-with-queen-and-without-queen

### Estructura declarada
- Carpeta **"queenbee absent"**: 2.000 archivos de 10 segundos — fecha: **31-05-2018**
- Carpeta **"queenbee present"**: 4.000 archivos de 10 segundos — fecha: **12-06-2018**

### ❌ Decisión: EXCLUIDO — Duplicado de NU-HIVE

**Evidencia de duplicación:**

| Característica | NU-HIVE (Colmena 1) | harshkumar1711 |
|---|---|---|
| Fecha QL | 31-05-2018 | "queenbee absent" → 31-05-2018 |
| Fecha QR | 12-06-2018 | "queenbee present" → 12-06-2018 |
| Duración clips | ~9 min 52 s | 10 segundos (= segmentados del mismo audio) |
| Colmena | Colmena 1 | No documentado |

Las fechas coinciden exactamente. harshkumar1711 tomó los audios de NU-HIVE Colmena 1, los segmentó en clips de 10 segundos y los subió a Kaggle sin citar la fuente original. Usar ambos datasets en el mismo experimento constituiría **data leakage** (el modelo vería las mismas grabaciones dos veces).

**Documentación:** Esta detección debe reportarse en la tesis como parte de la auditoría de datos. Es un ejemplo de la importancia de verificar la procedencia de datasets de Kaggle.

---

## 2.6. UrBAN 2021 — Dataset de Prueba Cross-Country

**Fuente:** Dataset privado de investigación  
**Período:** 2021 (agosto–octubre)  
**Ubicación:** Montreal, Quebec, Canadá

### Estructura de audio (resultado de UrBAN-03)
- **10 colmenas** con audio, **6 usables** (con sensor y audio simultáneos)
- Colmenas usables: 3627, 3631, 3690, 3691, 3692, 3693
- **Frecuencia:** 1 archivo cada 30 minutos (2 archivos por hora)
- **Duración por archivo:** ~14 min 47 s (~887 segundos)
- **Total:** 3.753 archivos, 922,5 horas de audio
- **Días con audio + sensor simultáneos:** 50 días (repartidos entre 6 colmenas)
- **Formato de nombre:** `DD-MM-YYYY_HHhMM_HIVE-{id}.wav`

### Etiquetado (resultado de UrBAN-02)
- **Sin etiqueta por archivo.** El estado de la colmena solo se conoce en las fechas de inspección del apicultor.
- El notebook UrBAN-02 construyó 4 columnas de contexto: `ultima_insp_fecha`, `ultima_insp_estado`, `proxima_insp_fecha`, `proxima_insp_estado`.
- Para xnorC, se usa la **inspección más cercana como referencia etiquetada** (ancla).

### Estrategia de uso para xnorC
1. Tomar la inspección del apicultor más cercana como muestra de referencia (QR o QL conocido)
2. Formar pares: (referencia, cada audio de la ventana de evaluación)
3. Predecir IGUAL/DIFERENTE con xnorC entrenado en Yang + NU-HIVE
4. Convertir a QR/QL usando la etiqueta de la referencia
5. Evaluar contra las inspecciones del apicultor disponibles

**Limitación conocida:** con solo 50 días de solapamiento audio+sensor y etiquetas escasas, los resultados serán indicativos, no estadísticamente concluyentes. Esto se documenta explícitamente en la tesis.

---

## 2.7. MSPB 2.0 — Dataset Referencia Argentina (Fase Posterior)

**Período:** 2022–2023  
**Ubicación:** Argentina

### Situación
MSPB no tiene archivos de audio crudo — las features de audio ya están calculadas y almacenadas en el dataset tabular como columnas `hz_XXX` (bins de FFT). Incluirlo en xnorC-AUD requeriría que todos los otros datasets extraigan las mismas features de la misma manera, lo que introduce riesgo de incompatibilidad.

### ❌ Excluido de esta fase (xnorC-AUD)
MSPB se incorporará en la **fase posterior** con xnorC-TAB (pares de variables de temperatura y humedad), donde sí hay compatibilidad directa con Yang y UrBAN.

---
# 3. Comparación de Datasets — Compatibilidad Técnica

Para construir un pipeline unificado, es importante entender las diferencias técnicas entre Yang y NU-HIVE.

In [ ]:
import pandas as pd

# Tabla comparativa de características técnicas de los datasets
comparacion = pd.DataFrame({
    'Característica': [
        'País',
        'Colmenas con audio',
        'Colmenas usables para xnorC',
        'Duración por archivo (seg)',
        'Intervalo entre archivos (min)',
        'Archivos por hora',
        'Tipo de etiquetado',
        'Clases disponibles',
        'Días de audio disponibles',
        'Total horas de audio (aprox)',
        'Sample rate (Hz)',
        'Canales',
        'Bits por muestra',
    ],
    'Yang (SBCM)': [
        'EE.UU. (California)',
        '4',
        '4',
        '60',
        '10',
        '6',
        'Continuo por hora',
        'QR / QL',
        'Semanas continuas',
        '~1.000+',
        'Por confirmar (OE2B-02)',
        'Por confirmar (OE2B-02)',
        'Por confirmar (OE2B-02)',
    ],
    'NU-HIVE': [
        'Europa (UK)',
        '2',
        '2',
        '592 (~9m52s)',
        '10',
        '~6',
        'Por día completo',
        'QR / QL',
        '4 días (1 por clase por colmena)',
        '~24 (4 días × 6h útiles)',
        'Por confirmar (OE2B-01)',
        'Por confirmar (OE2B-01)',
        'Por confirmar (OE2B-01)',
    ],
    'UrBAN 2021 (prueba)': [
        'Canadá (Montreal)',
        '10 (6 usables)',
        '6',
        '887 (~14m47s)',
        '30',
        '2',
        'Solo en fechas de inspección',
        'QR / QL (puntual)',
        '50 días con audio+sensor',
        '922,5',
        '16.000',
        'Mono (1)',
        '16',
    ],
})

print("Comparación técnica de datasets:")
print()
print(comparacion.to_string(index=False))

## 3.1. Plan de estandarización

Para que los tres datasets sean comparables, se aplicará el siguiente preprocesamiento en OE2B-01 y OE2B-02:

| Paso | Decisión | Justificación |
|---|---|---|
| **Sample rate** | Resamplear todo a **8.000 Hz** | Es el mínimo común divisor útil para audio de colmenas. Elimina el contenido sobre 4 kHz que no es relevante para el zumbido de abejas. Compatible con la literatura (BeeTogether usa 8 kHz). |
| **Canales** | Convertir a **mono** (si no lo es ya) | UrBAN ya es mono. Yang y NU-HIVE por confirmar. |
| **Normalización de amplitud** | Normalizar a rango **[-1, +1]** por archivo | Elimina diferencias de ganancia entre micrófonos. |
| **Segmentación** | Ventanas de duración fija según el experimento | Ver sección 4 (experimentos). |
| **Granularidad temporal** | Agregar features a **1 hora** | Consistente con sensores y con el plan de OE1. |

## 3.2. Desafío de los archivos largos de UrBAN

UrBAN tiene archivos de ~14 min 47 s. Yang tiene archivos de 1 min. Para los experimentos con ventanas cortas (2s, 10s), los archivos de UrBAN se **segmentarán** en clips del tamaño requerido. Esto se hace en OE2B-07 (evaluación cross-country), no en el preprocesamiento general.

---
# 4. Experimentos Planificados

## 4.1. Visión general

Todos los experimentos de esta fase usan **solo modalidad de audio** (xnorC-AUD). Los experimentos de modalidad tabular (xnorC-TAB) y multimodal (xnorC-HYB) quedan para una fase posterior.

| ID | Nombre | Representación | Entrenamiento | Validación | Propósito |
|---|---|---|---|---|---|
| **B0** | ClassiC-AUD (Mel) | Espectrograma Mel | Yang + NU-HIVE | k-hive-fold | Línea base clásica — muestra el gap de generalización |
| **B1** | xnorC-AUD (Mel) | Espectrograma Mel | Yang + NU-HIVE | k-hive-fold | Replicación BeeTogether — línea base del enfoque de pares |
| **B2** | xnorC-AUD (MFCC) | MFCC (13 coef. + deltas) | Yang + NU-HIVE | k-hive-fold | Ablation: representación compacta del timbre |
| **B3** | xnorC-AUD (FFT) | Bins de FFT (~30 Hz) | Yang + NU-HIVE | k-hive-fold | Ablation: representación compatible con MSPB |
| **B4** | xnorC-AUD (CQT) | Constant-Q Transform | Yang + NU-HIVE | k-hive-fold | Ablation: sensible a armónicos del zumbido |
| **B5** | Mejor xnorC → UrBAN | La mejor de B1–B4 | Yang + NU-HIVE | UrBAN 2021 | Evaluación cross-country |

> **Nota:** B0 (ClassiC) se incluye deliberadamente para demostrar el gap de generalización y justificar el enfoque xnorC. Es la "línea base mala" que motiva todo el trabajo.

---

## 4.2. Experimento B0 — ClassiC-AUD (Mel)

**Propósito:** demostrar el gap de generalización del clasificador clásico. Es la motivación para usar xnorC.

**Arquitectura:**
- Entrada: espectrograma Mel (128 bandas, ventana de 2 segundos)
- Red: CNN 3 capas convolucionales + 2 capas densas
- Salida: sigmoide binaria (QR / QL)

**Entrenamiento:**
- Datos: segmentos individuales de Yang + NU-HIVE con etiquetas directas
- Validación: k-hive-fold (una colmena entera excluida por fold)
- Métrica: F1-macro

**Resultado esperado:** F1 alto (~0.90+) en validación aleatoria, F1 cercano a 0.50 en k-hive-fold. Este contraste es el que justifica xnorC.

---

## 4.3. Experimento B1 — xnorC-AUD (Mel) — Línea base

**Propósito:** replicación de BeeTogether con nuestros datasets. Es la línea base del enfoque de pares.

**Representación de audio:** Espectrograma Mel
- Frecuencia de muestreo: 8.000 Hz
- Ventana FFT: 512 muestras (64 ms)
- Hop length: 160 muestras (20 ms)
- Bandas Mel: 128
- Escala: dB (log)
- Duración de segmento: 2 segundos (igual que BeeTogether)

**Arquitectura xnorC:**
- **Rama compartida (encoder):** CNN sobre espectrograma → embedding de 128 dimensiones
  - Conv2D(32, 3×3) + BatchNorm + ReLU + MaxPool
  - Conv2D(64, 3×3) + BatchNorm + ReLU + MaxPool
  - Conv2D(128, 3×3) + BatchNorm + ReLU + GlobalAvgPool
  - Dense(128) + ReLU + Dropout(0.3)
- **Fusión:** concatenación de los dos embeddings del par → diferencia elemento a elemento
- **Cabeza:** Dense(64) + ReLU + Dense(1) + Sigmoide
- **Salida:** IGUAL (1) / DIFERENTE (0)

**Entrenamiento:**
- Pérdida: binary cross-entropy
- Optimizador: Adam (lr=1e-3, reducción en plateau)
- Early stopping: paciencia 10 épocas sobre F1 de validación
- Batch size: 64 pares
- Augmentación: pitch shift ±2 semitonos, ruido gaussiano (SNR 20 dB), SpecAugment

**Construcción de pares:**
- N = 500 pares por colmena (250 IGUAL + 250 DIFERENTE)
- Pares DIFERENTE: priorizados cerca del momento de cambio de estado
- Pares IGUAL: de períodos estables (alejados del cambio de estado)
- **REGLA CRÍTICA:** nunca se forman pares entre colmenas distintas

**Validación:**
- k-hive-fold: k = número de colmenas en entrenamiento
- Métrica: F1-macro sobre la clasificación IGUAL/DIFERENTE
- Métrica secundaria: F1-macro sobre QR/QL inferido (usando referencia)

---

## 4.4. Experimento B2 — xnorC-AUD (MFCC)

**Propósito:** evaluar si una representación compacta del timbre (MFCC) generaliza mejor que el espectrograma completo.

**Representación de audio:** MFCC + Delta-MFCC
- Coeficientes MFCC: 13
- Delta (velocidad): 13 coeficientes derivados
- Delta-delta (aceleración): 13 coeficientes
- Total: 39 features por ventana temporal
- Duración de segmento: 2 segundos

**Arquitectura xnorC:**
- Rama compartida: MLP sobre vector de 39 features (en lugar de CNN sobre imagen)
  - Dense(128) + ReLU + Dropout(0.3)
  - Dense(128) + ReLU + Dropout(0.3)
  - Dense(64) + ReLU → embedding de 64 dimensiones
- Fusión y cabeza: igual que B1

**Todo lo demás** (construcción de pares, validación, métricas) igual que B1.

---

## 4.5. Experimento B3 — xnorC-AUD (FFT bins)

**Propósito:** usar la misma representación que ya tiene MSPB (bins de FFT con resolución de ~30 Hz). Esto permitiría en el futuro aplicar el modelo directamente a MSPB sin recalcular features.

**Representación de audio:** Bins de FFT
- Resolución: ~30 Hz por bin
- Ventana FFT: igual que la usada en MSPB (a confirmar en auditoría MSPB)
- Rango de frecuencias: 0 – 4.000 Hz (con sample rate de 8 kHz)
- Total bins: ~133 bins de frecuencia
- Magnitud promediada en la ventana temporal

**Arquitectura xnorC:**
- Rama compartida: igual que B2 (MLP) pero sobre vector de ~133 bins
- Fusión y cabeza: igual que B1

**Todo lo demás** igual que B1.

---

## 4.6. Experimento B4 — xnorC-AUD (CQT)

**Propósito:** evaluar si la Constant-Q Transform, sensible a armónicos y frecuencias musicales, captura mejor el zumbido periódico de la colmena que el espectrograma Mel lineal.

**Representación de audio:** CQT (Constant-Q Transform)
- Bins por octava: 36
- Rango: C1 (32 Hz) – C7 (2.048 Hz)
- Escala: dB (log)
- Duración de segmento: 2 segundos

**Hipótesis:** el zumbido de las abejas tiene componentes armónicas regulares. CQT está diseñado para ser invariante a traslaciones en la escala logarítmica de frecuencias, lo que podría hacerlo más robusto a variaciones de temperatura entre datasets.

**Arquitectura xnorC:** igual que B1 (CNN sobre imagen tiempo-frecuencia).

**Todo lo demás** igual que B1.

---

## 4.7. Experimento B5 — Evaluación Cross-Country (UrBAN)

**Propósito:** evaluar la generalización de xnorC en un contexto cross-country real. Es el experimento de mayor valor científico de OE2-B.

**Configuración:**
- Modelo: el mejor de B1–B4 según F1 en k-hive-fold
- Entrenamiento: Yang + NU-HIVE (sin ningún dato de UrBAN)
- Prueba: colmenas 3627, 3631, 3690, 3691, 3692, 3693 de UrBAN 2021

**Protocolo de evaluación:**
1. Para cada colmena de UrBAN: tomar la inspección del apicultor más cercana como referencia
2. Formar pares: (referencia, cada audio en ventana ±7 días alrededor de cada inspección)
3. Predecir IGUAL/DIFERENTE con xnorC
4. Convertir a QR/QL usando la etiqueta de la referencia
5. Comparar con la próxima inspección del apicultor
6. Calcular F1-macro, precisión, recall

**Métricas adicionales:**
- **Días de anticipación:** si el modelo predice QL antes de que el apicultor lo confirme, ¿cuántos días antes?
- **Tasa de detección de cambio:** ¿en qué porcentaje de transiciones QR→QL el modelo detecta el cambio?

**Limitación documentada:** con 50 días de audio+sensor y etiquetas escasas (solo en fechas de inspección), los resultados son indicativos. Se reportarán con intervalos de confianza por bootstrap.

---
# 5. Construcción del Dataset de Pares — Reglas Críticas

La construcción del dataset de pares es el paso más crítico de todo OE2-B. Un error aquí invalida todos los experimentos.

## 5.1. Definición de un par

Un par consiste en dos segmentos de audio de la **misma colmena**, tomados en momentos distintos.

```
par = (segmento_A, segmento_B, etiqueta_par)

etiqueta_par = IGUAL     si estado(A) == estado(B)   [QR-QR o QL-QL]
etiqueta_par = DIFERENTE si estado(A) != estado(B)   [QR-QL o QL-QR]
```

## 5.2. Reglas de construcción

| Regla | Descripción | Por qué |
|---|---|---|
| **R1 — Misma colmena** | Ambos elementos del par pertenecen a la MISMA colmena | Si se mezclan colmenas, el modelo aprende diferencias entre colmenas, no diferencias de estado |
| **R2 — Sin mezcla train/test** | Las colmenas de prueba no aparecen en NINGÚN par de entrenamiento | Evita data leakage y garantiza validación real de generalización |
| **R3 — Balance de clases** | 50% pares IGUAL, 50% pares DIFERENTE | Evita sesgo del modelo hacia la clase mayoritaria |
| **R4 — Pares DIFERENTE cerca del cambio** | Los pares DIFERENTE se forman con al menos un elemento cerca de la transición QR→QL | El modelo aprende los cambios sutiles, no solo diferencias obvias |
| **R5 — Pares IGUAL en zonas estables** | Los pares IGUAL se forman con ambos elementos lejos de la transición | Evita incluir períodos ambiguos como "igual" |

## 5.3. Verificación explícita (a implementar en OE2B-03)

El notebook OE2B-03 incluirá una celda de verificación que:
1. Confirma que ninguna colmena aparece en train Y en test simultáneamente
2. Verifica que todos los pares de train tienen ambos elementos de la misma colmena
3. Verifica el balance de clases (proporción IGUAL/DIFERENTE)
4. Muestra ejemplos de pares IGUAL y DIFERENTE con sus timestamps

## 5.4. Colmenas disponibles y distribución

| Dataset | Colmenas | Uso en k-hive-fold |
|---|---|---|
| Yang | 4 colmenas | 4 folds: cada fold excluye 1 colmena de Yang |
| NU-HIVE | 2 colmenas | Se suman a Yang: total 6 colmenas → 6 folds |

Con 6 colmenas en total (4 Yang + 2 NU-HIVE), el k-hive-fold tiene 6 iteraciones. En cada iteración, 5 colmenas se usan para entrenar y 1 para validar.

---
# 6. Aportes Originales respecto a BeeTogether

| Aspecto | BeeTogether (2024) | Este trabajo |
|---|---|---|
| Representación audio | Solo espectrograma Mel | Mel + MFCC + FFT bins + CQT (ablation) |
| Modalidades | Solo audio | Audio (esta fase) + tabular + híbrido (fase posterior) |
| Nivel de generalización | Cross-hive (mismo dataset) | Cross-hive + **cross-country** |
| Datasets de prueba | NUHIVE, SBCM, BAD (mismo origen geográfico) | UrBAN (Canadá), MSPB (Argentina) |
| Ventana temporal | Fija: 2 segundos | 2 s (esta fase); ablation de ventanas en fase posterior |
| Contexto geográfico | América del Norte + Europa | + Argentina + Canadá (clima subártico) |

## 6.1. Contribución principal

El aporte más original de este trabajo es la **evaluación cross-country**: aplicar xnorC entrenado en colmenas de EE.UU. y UK sobre colmenas de Montreal (clima continental frío) y eventualmente Argentina (clima subtropical). BeeTogether demostró generalización cross-hive dentro de un mismo dataset; este trabajo extiende esa evaluación a colmenas de países diferentes con condiciones climáticas distintas.

## 6.2. Respuestas anticipadas al tutor/jurado

**"¿Esto no es una copia de BeeTogether?"**  
BeeTogether demostró que xnorC funciona en cross-hive dentro de un dataset. Este trabajo evalúa si eso se mantiene al cruzar países y condiciones climáticas, usando 4 representaciones acústicas distintas (ablation que BeeTogether no hizo), e incorporando modalidad ambiental (fase posterior).

**"¿Qué pasa si xnorC tampoco generaliza cross-country?"**  
Ese resultado es igualmente valioso: identificaría que la generalización de xnorC tiene un límite geográfico/climático, acotando el alcance de las técnicas existentes y abriendo una pregunta de investigación para trabajo futuro.

**"¿Por qué no aprendizaje contrastativo (SimCLR)?"**  
SimCLR requiere grandes volúmenes de datos no etiquetados. Los datasets de audio de colmenas son pequeños y tienen etiquetas disponibles; xnorC las aprovecha de manera más eficiente. SimCLR se exploraría si no hubiera etiquetas en absoluto.

---
# 7. Riesgos y Mitigaciones

| Riesgo | Probabilidad | Impacto | Mitigación |
|---|---|---|---|
| Data leakage en construcción de pares | Media | Alto | Verificación explícita en OE2B-03 con assertions de código |
| Pocas muestras QL en UrBAN | Alta | Medio | Reportar con bootstrap; etiquetar como "resultado indicativo" |
| Incompatibilidad de sample rate entre datasets | Media | Medio | Resamplear todo a 8 kHz antes de cualquier extracción |
| xnorC no generaliza cross-country | Media | Bajo (el hallazgo es válido) | Reportar como resultado negativo significativo |
| NU-HIVE tiene muy pocos días de datos | Alta | Medio | Aumentación de datos agresiva; reportar limitación |

## 7.1. Limitación más importante

**NU-HIVE tiene solo 4 días de grabación** (1 día por clase por colmena). Esto limita la variedad de condiciones que el modelo ve durante entrenamiento. La augmentación (pitch shift, ruido, SpecAugment) es especialmente importante para NU-HIVE.

---
# 8. Cronograma — Próximas 2 Semanas

Objetivo: tener resultados de B0 y B1 (ClassiC vs. xnorC en k-hive-fold) para mostrar al tutor.

| Semana | Día | Tarea | Notebook |
|---|---|---|---|
| **1** | 1–2 | Auditoría NU-HIVE (formato, sample rate, estructura) | OE2B-01 |
| **1** | 3–4 | Auditoría Yang para Parte B (confirmar sample rate, formato, colmenas) | OE2B-02 |
| **1** | 5–7 | Preprocesamiento: resamplear a 8 kHz, normalizar, segmentar | OE2B-01/02 |
| **2** | 8–9 | Construcción dataset de pares + verificación de integridad | OE2B-03 |
| **2** | 10–11 | Extracción de features (Mel para B1, MFCC para B2) | OE2B-04 |
| **2** | 12–13 | Entrenamiento B0 (ClassiC) + B1 (xnorC-Mel) | OE2B-05 |
| **2** | 14 | Evaluación k-hive-fold + comparación B0 vs. B1 | OE2B-06 |

**Resultado mínimo para el tutor:** tabla comparativa de F1 en validación aleatoria vs. k-hive-fold para ClassiC (B0) y xnorC-Mel (B1). Eso solo muestra el gap de generalización y justifica el enfoque.

---
# 9. Resumen de Decisiones

Esta sección registra las decisiones de diseño tomadas durante la planificación, con su justificación. Es útil para responder preguntas del tutor y para escribir la sección de metodología de la tesis.

| # | Decisión | Alternativa considerada | Justificación |
|---|---|---|---|
| D1 | Excluir harshkumar1711 | Usarlo como dataset adicional | Es una re-publicación de NU-HIVE sin citar fuente — usarlo generaría data leakage |
| D2 | Excluir OSBH | Usar solo la clase "Active" | 1 solo archivo de clase negativa no permite entrenar ni validar |
| D3 | Resamplear a 8 kHz | Mantener sample rate original de cada dataset | Estandarización necesaria para comparar features entre datasets |
| D4 | Agregar features a 1 hora | Mantener granularidad de archivo | Consistencia con sensores ambientales y con la estrategia de OE1 |
| D5 | Excluir MSPB de xnorC-AUD | Calcular FFT bins compatibles | Riesgo de incompatibilidad de features; MSPB va en xnorC-TAB (fase posterior) |
| D6 | UrBAN como prueba (no entrenamiento) | Usar UrBAN en entrenamiento con etiquetas parciales | El objetivo es evaluar generalización, no aumentar datos de entrenamiento |
| D7 | N=500 pares por colmena | Más o menos pares | Punto de partida de BeeTogether; ajustar si el modelo no converge |
| D8 | Ventana de 2 segundos para esta fase | Ventanas de 10s, 30s, 60s | Consistencia con BeeTogether para comparación directa; ablation de ventanas en fase posterior |

In [ ]:
# ─── Checklist de estado del pipeline ────────────────────────────────────────
# Actualizá este diccionario a medida que avanzás

estado_pipeline = {
    # Auditorías
    'OE2B-00  Plan maestro':                         '✅ Completo',
    'OE2B-01  Auditoría NU-HIVE':                    '⬜ Pendiente',
    'OE2B-02  Auditoría Yang (Parte B)':             '⬜ Pendiente',
    # Preprocesamiento
    'OE2B-03  Construcción dataset de pares':        '⬜ Pendiente',
    'OE2B-04  Extracción de features':               '⬜ Pendiente',
    # Modelos
    'OE2B-05  Entrenamiento xnorC':                  '⬜ Pendiente',
    'OE2B-06  Evaluación cross-hive (k-hive-fold)':  '⬜ Pendiente',
    'OE2B-07  Evaluación cross-country (UrBAN)':     '⬜ Pendiente',
    # Experimentos específicos
    'B0  ClassiC-AUD (Mel)':                         '⬜ Pendiente',
    'B1  xnorC-AUD (Mel)':                           '⬜ Pendiente',
    'B2  xnorC-AUD (MFCC)':                          '⬜ Pendiente',
    'B3  xnorC-AUD (FFT bins)':                      '⬜ Pendiente',
    'B4  xnorC-AUD (CQT)':                           '⬜ Pendiente',
    'B5  Mejor xnorC → UrBAN':                       '⬜ Pendiente',
}

print("Estado del pipeline OE2-B:")
print()
for tarea, estado in estado_pipeline.items():
    print(f"  {estado}  {tarea}")